# OpenOximetry Reference Stream Audit

Audit continuous oxygen-reference viability for timing-eligible encounters.

Inputs from: `outputs/openox_ppg_eligible_encounters.csv` (from notebook 06).
Outputs to: `outputs/openox_reference_stream_coverage.csv` and `outputs/openox_reference_encounter_passlist.csv`.
Next: notebook 10 for canonical feature export on the passlist.

## Load Eligible Encounter Set

Load timing-eligible encounters and initialize output locations for reference-stream coverage artifacts.

In [ ]:
from pathlib import Path
import csv

import numpy as np
import pandas as pd


def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "outputs" / "openox_ppg_eligible_encounters.csv").is_file() and (candidate / "notebooks").is_dir():
            return candidate
    raise RuntimeError("Could not find project root containing outputs/openox_ppg_eligible_encounters.csv and notebooks/")


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
OUTPUTS_DIR = PROJECT_ROOT / "outputs"
ELIGIBLE_CSV = OUTPUTS_DIR / "openox_ppg_eligible_encounters.csv"
STREAM_COVERAGE_CSV = OUTPUTS_DIR / "openox_reference_stream_coverage.csv"
PASSLIST_CSV = OUTPUTS_DIR / "openox_reference_encounter_passlist.csv"

eligible_df = pd.read_csv(ELIGIBLE_CSV)
print(f"Project root: {PROJECT_ROOT}")
print(f"Eligible encounters: {len(eligible_df)}")

## Evaluate Oxygen-Reference Candidates

Inspect 2Hz files, choose the best available oxygen-reference column, and measure non-null coverage and value ranges per encounter.

In [ ]:
REFERENCE_CANDIDATES = (
    "ScalcO2",
    "SpO2",
    "Saturation",
    "sat",
)


def choose_reference_column(fieldnames: list[str]) -> str | None:
    lookup = {name.lower(): name for name in fieldnames}
    for candidate in REFERENCE_CANDIDATES:
        match = lookup.get(candidate.lower())
        if match is not None:
            return match
    return None


coverage_rows = []
for row in eligible_df.itertuples(index=False):
    hz2_path = Path(row.hz2_csv_path)
    with hz2_path.open("r", encoding="utf-8", errors="ignore", newline="") as handle:
        reader = csv.DictReader(handle)
        fieldnames = reader.fieldnames or []
        reference_column = choose_reference_column(fieldnames)
        non_null_count = 0
        total_rows = 0
        values = []
        for hz_row in reader:
            total_rows += 1
            if reference_column is None:
                continue
            value = pd.to_numeric(hz_row.get(reference_column), errors="coerce")
            if pd.notna(value):
                non_null_count += 1
                values.append(float(value))

    coverage_rows.append(
        {
            "encounter_id": str(row.encounter_id),
            "hz2_csv_path": str(hz2_path),
            "reference_column": reference_column,
            "hz2_total_rows": int(total_rows),
            "reference_non_null_rows": int(non_null_count),
            "reference_non_null_ratio": float(non_null_count / total_rows) if total_rows else 0.0,
            "reference_min": float(np.min(values)) if values else np.nan,
            "reference_max": float(np.max(values)) if values else np.nan,
        }
    )

coverage_df = pd.DataFrame(coverage_rows).sort_values("encounter_id").reset_index(drop=True)
coverage_df.to_csv(STREAM_COVERAGE_CSV, index=False)
coverage_df.head()

## Build Canonical Passlist

Filter encounters using minimum non-null reference coverage thresholds and export the canonical passlist for downstream feature extraction.

In [ ]:
passlist_df = coverage_df.loc[
    coverage_df["reference_column"].notna()
    & (coverage_df["reference_non_null_rows"] >= 20)
    & (coverage_df["reference_non_null_ratio"] >= 0.25)
].copy()

passlist_df.to_csv(PASSLIST_CSV, index=False)

print(f"Reference coverage rows: {len(coverage_df)}")
print(f"Canonical reference passlist rows: {len(passlist_df)}")
passlist_df.head()